## 1. Import Libraries

We import `pandas` and inject our project root into `sys.path` so we can natively import our custom `DataNormalizer` module from the ETL pipeline.

In [1]:
import pandas as pd
from pathlib import Path
import sys

current_dir = Path.cwd()

# Dynamically resolve the project root
if current_dir.name == "notebooks":
    PROJECT_ROOT = current_dir.parent
else:
    PROJECT_ROOT = current_dir

# Allow imports from 'src'
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.etl.normalizer import DataNormalizer

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 200)

print("Libraries Imported Successfully")

Libraries Imported Successfully


## 2. Project Paths

We configure dynamic paths to automatically find our `raw` data, `supporting` data, and establish a new `processed` data directory for normalized files.

In [2]:
RAW_DATA = PROJECT_ROOT / "data" / "raw"
SUPPORTING_DATA = PROJECT_ROOT / "data" / "supporting"
PROCESSED_DATA = PROJECT_ROOT / "data" / "processed"

# Ensure processed data folder exists
PROCESSED_DATA.mkdir(exist_ok=True, parents=True)

print("Project Root:", PROJECT_ROOT)

Project Root: c:\Users\polai\OneDrive\Desktop\N100 FINANCIAL INTELLIGENCE PLATFORM\nifty100-financial-intelligence-platform


## 3. Find All Datasets

We recursively collect all Excel files from both the raw and supporting data directories to build our processing queue.

In [3]:
raw_files = sorted(RAW_DATA.glob("*.xlsx"))
supporting_files = sorted(SUPPORTING_DATA.glob("*.xlsx"))

excel_files = raw_files + supporting_files

print(f"Datasets Found : {len(excel_files)}")

for file in excel_files:
    print(file.name)

Datasets Found : 12
analysis.xlsx
balancesheet.xlsx
cashflow.xlsx
companies.xlsx
documents.xlsx
profitandloss.xlsx
prosandcons.xlsx
financial_ratios.xlsx
market_cap.xlsx
peer_groups.xlsx
sectors.xlsx
stock_prices.xlsx


## 4. Test One Dataset

We first load the master `companies.xlsx` file to test the normalizer and review its initial data types and structure.

In [4]:
try:
    df = pd.read_excel(RAW_DATA / "companies.xlsx")
    display(df.head())
    print("\n--- Dtypes ---")
    print(df.dtypes)
except Exception as e:
    print(f"Error reading file: {e}")

,Mkt Fintech — Nifty 100 | Companies | 92 records,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11
0,id,company_logo,company_name,chart_link,about_company,website,nse_profile,bse_profile,face_value,book_value,roce_percentage,roe_percentage
1,ABB,https://mkt.in/static/mkt-icons/nifty100/ABB.png,Abbott India Ltd,https://in.tradingview.com/chart/?symbol=NSE%3...,Abbott India Ltd is one of the leading multina...,https://www.abbott.co.in/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/abb...,10,1657,46,34.9
2,ADANIENSOL,https://m.economictimes.com/thumb/msid-1173715...,Adani Energy Solutions Ltd,https://in.tradingview.com/chart/?symbol=NSE%3...,"AESL, part of the Adani portfolio, is a multid...",https://www.adanienergysolutions.com/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/ada...,10,175,9,8.59
3,ADANIENT,https://mkt.in/static/mkt-icons/nifty100/ADANI...,Adani Enterprises Ltd,https://in.tradingview.com/chart/?symbol=ADANIENT,Adani Enterprises Ltd is an Indian multination...,https://www.adanienterprises.com/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/ada...,1,363,11.6,13.64
4,ADANIGREEN,https://mkt.in/static/mkt-icons/nifty100/ADANI...,Adani Green Energy Ltd,https://in.tradingview.com/chart/?symbol=NSE%3...,"Adani Green Energy Limited, incorporated in 20...",http://www.adanigreenenergy.com/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/ada...,10,67,96.5,14.7



--- Dtypes ---
Mkt Fintech — Nifty 100  |  Companies  |  92 records       str
Unnamed: 1                                                 str
Unnamed: 2                                                 str
Unnamed: 3                                                 str
Unnamed: 4                                                 str
Unnamed: 5                                                 str
Unnamed: 6                                                 str
Unnamed: 7                                                 str
Unnamed: 8                                              object
Unnamed: 9                                              object
Unnamed: 10                                             object
Unnamed: 11                                             object
dtype: object


## 5. Apply Normalizer

We run the dataframe through our `DataNormalizer` pipeline, scrubbing bad characters, casting types, and replacing string nulls with `NaN`.

In [5]:
if 'df' in locals():
    normalized_df = DataNormalizer.normalize(df.copy())
    display(normalized_df.head())

,Mkt Fintech — Nifty 100 | Companies | 92 records,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11
0,id,company_logo,company_name,chart_link,about_company,website,nse_profile,bse_profile,NaN,NaN,NaN,NaN
1,ABB,https://mkt.in/static/mkt-icons/nifty100/ABB.png,Abbott India Ltd,https://in.tradingview.com/chart/?symbol=NSE%3...,Abbott India Ltd is one of the leading multina...,https://www.abbott.co.in/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/abb...,10.0,1657.0,46.0,34.90
2,ADANIENSOL,https://m.economictimes.com/thumb/msid-1173715...,Adani Energy Solutions Ltd,https://in.tradingview.com/chart/?symbol=NSE%3...,"AESL, part of the Adani portfolio, is a multid...",https://www.adanienergysolutions.com/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/ada...,10.0,175.0,9.0,8.59
3,ADANIENT,https://mkt.in/static/mkt-icons/nifty100/ADANI...,Adani Enterprises Ltd,https://in.tradingview.com/chart/?symbol=ADANIENT,Adani Enterprises Ltd is an Indian multination...,https://www.adanienterprises.com/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/ada...,1.0,363.0,11.6,13.64
4,ADANIGREEN,https://mkt.in/static/mkt-icons/nifty100/ADANI...,Adani Green Energy Ltd,https://in.tradingview.com/chart/?symbol=NSE%3...,"Adani Green Energy Limited, incorporated in 20...",http://www.adanigreenenergy.com/,https://www.nseindia.com/get-quotes/equity?sym...,https://www.bseindia.com/stock-share-price/ada...,10.0,67.0,96.5,14.70


## 6. Compare Before & After

We compare the data types before and after normalization to ensure our pipeline successfully converted strings to numerics and integers where appropriate.

In [ ]:
if 'df' in locals() and 'normalized_df' in locals():
    comparison = pd.DataFrame({
        "Column": df.columns,
        "Before (Dtype)": df.dtypes.astype(str).values,
        "After (Dtype)": normalized_df.dtypes.astype(str).values
    })
    display(comparison)

,Column,Before (Dtype),After (Dtype)
0,Mkt Fintech — Nifty 100 | Companies | 92 r...,str,str
1,Unnamed: 1,str,str
2,Unnamed: 2,str,str
3,Unnamed: 3,str,str
4,Unnamed: 4,str,str
5,Unnamed: 5,str,str
6,Unnamed: 6,str,str
7,Unnamed: 7,str,str
8,Unnamed: 8,object,float64
9,Unnamed: 9,object,float64


## 7. Check Missing Values

We verify that our normalizer correctly identified ghost-nulls (like `"-"`, `"NA"`) and converted them to standard pandas nulls.

In [7]:
if 'df' in locals() and 'normalized_df' in locals():
    before = df.isnull().sum()
    after = normalized_df.isnull().sum()

    missing = pd.DataFrame({
        "Column": df.columns,
        "Nulls Before": before.values,
        "Nulls After": after.values
    })
    display(missing)

,Column,Nulls Before,Nulls After
0,Mkt Fintech — Nifty 100 | Companies | 92 r...,0,0
1,Unnamed: 1,1,0
2,Unnamed: 2,0,0
3,Unnamed: 3,0,0
4,Unnamed: 4,0,0
5,Unnamed: 5,1,0
6,Unnamed: 6,1,0
7,Unnamed: 7,1,0
8,Unnamed: 8,1,2
9,Unnamed: 9,1,2


## 8. Normalize Every Dataset

We loop through all discovered Excel files, normalize them completely, and save the scrubbed output into the `processed` data folder as CSV files.

In [8]:
summary = []

for file in excel_files:
    try:
        print("=" * 80)
        print(f"Processing: {file.name}")

        # Extract
        temp_df = pd.read_excel(file)
        rows_before = len(temp_df)

        # Transform / Normalize
        temp_df = DataNormalizer.normalize(temp_df)
        rows_after = len(temp_df)

        # Load / Save
        output = PROCESSED_DATA / f"{file.stem}.csv"
        temp_df.to_csv(output, index=False)

        # Compile metrics
        summary.append({
            "Dataset": file.name,
            "Rows": rows_after,
            "Columns": len(temp_df.columns),
            "Missing Values": int(temp_df.isnull().sum().sum()),
            "Output": output.name
        })

        print(f"Saved -> {output.name}")
    except Exception as e:
        print(f"Failed to process {file.name}: {e}")

Processing: analysis.xlsx
Saved -> analysis.csv
Processing: balancesheet.xlsx
Saved -> balancesheet.csv
Processing: cashflow.xlsx
Saved -> cashflow.csv
Processing: companies.xlsx
Saved -> companies.csv
Processing: documents.xlsx
Saved -> documents.csv
Processing: profitandloss.xlsx
Saved -> profitandloss.csv
Processing: prosandcons.xlsx
Saved -> prosandcons.csv
Processing: financial_ratios.xlsx
Saved -> financial_ratios.csv
Processing: market_cap.xlsx
Saved -> market_cap.csv
Processing: peer_groups.xlsx
Saved -> peer_groups.csv
Processing: sectors.xlsx
Saved -> sectors.csv
Processing: stock_prices.xlsx
Saved -> stock_prices.csv


## 9. Summary Report

We assemble the processing metrics into a unified dataframe for review.

In [9]:
summary_df = pd.DataFrame(summary)
display(summary_df)

,Dataset,Rows,Columns,Missing Values,Output
0,analysis.xlsx,21,6,0,analysis.csv
1,balancesheet.xlsx,1313,13,11,balancesheet.csv
2,cashflow.xlsx,1188,7,13,cashflow.csv
3,companies.xlsx,93,12,9,companies.csv
4,documents.xlsx,1586,4,2,documents.csv
5,profitandloss.xlsx,1277,15,244,profitandloss.csv
6,prosandcons.xlsx,17,4,1,prosandcons.csv
7,financial_ratios.xlsx,1184,16,1354,financial_ratios.csv
8,market_cap.xlsx,552,9,0,market_cap.csv
9,peer_groups.xlsx,56,4,0,peer_groups.csv


## 10. Export Report

We export the normalization report to the `reports` directory so the data quality team can review it.

In [10]:
REPORT_PATH = PROJECT_ROOT / "reports"
REPORT_PATH.mkdir(exist_ok=True)

if not summary_df.empty:
    summary_df.to_csv(
        REPORT_PATH / "normalization_report.csv",
        index=False
    )
    print("Normalization Report Saved Successfully")
else:
    print("No data to save.")

Normalization Report Saved Successfully


## 11. Dataset Statistics

We print out aggregate statistics representing the scale and cleanliness of the processed data warehouse.

In [11]:
print("=" * 70)
print("NORMALIZATION SUMMARY")
print("=" * 70)

if not summary_df.empty:
    print(f"Datasets : {len(summary_df)}")
    print(f"Total Rows : {summary_df['Rows'].sum()}")
    print(f"Total Columns : {summary_df['Columns'].sum()}")
    print(f"Total Missing Values : {summary_df['Missing Values'].sum()}")

NORMALIZATION SUMMARY
Datasets : 12
Total Rows : 12899
Total Columns : 105
Total Missing Values : 1634


## 12. Preview Processed Files

We confirm the existence of all successfully normalized and exported CSV files inside our `processed` data directory.

In [ ]:
processed_files = sorted(PROCESSED_DATA.glob("*.csv"))

print(f"Processed Files : {len(processed_files)}")

for file in processed_files:
    print(file.name)

Processed Files : 12
analysis.csv
balancesheet.csv
cashflow.csv
companies.csv
documents.csv
financial_ratios.csv
market_cap.csv
peer_groups.csv
profitandloss.csv
prosandcons.csv
sectors.csv
stock_prices.csv
